# NYC Airbnb Recommendation System: Sentiment Analysis

This notebook closes the loop on the project by analyzing what guests 
actually say in their reviews, and checking whether that sentiment adds 
anything beyond what the structured data in Notebook 1's A/B test 
already found.

1. Load and clean guest review text
2. Run sentiment scoring using HuggingFace's DistilBERT model
3. Compare sentiment against occupancy and Instant Book status
4. Pull out common praise and complaint themes from the review text
5. Test whether sentiment adds real predictive signal on top of the 
   Project 1 variables
6. Write up the business recommendation

## Imports

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# load data
reviews = pd.read_csv('/Users/eggy/Downloads/reviews.csv.gz', compression='gzip')
listings_clean = pd.read_csv('data/listings_clean.csv')

print(f'Reviews: {reviews.shape}')
print(f'Listings: {listings_clean.shape}')

Reviews: (977459, 6)
Listings: (36613, 23)


## EDA

In [50]:
# basic stats
print('Reviews per listing:')
print(reviews.groupby('listing_id')['id'].count().describe())

print('\nDate range:')
print(f'Earliest review: {reviews["date"].min()}')
print(f'Latest review: {reviews["date"].max()}')

print('\nMissing comments:')
print(reviews['comments'].isna().sum())

print('\nSample reviews:')
print(reviews['comments'].dropna().sample(5, random_state=42).tolist())

Reviews per listing:
count    25093.000000
mean        38.953453
std         79.458967
min          1.000000
25%          3.000000
50%         11.000000
75%         42.000000
max       3518.000000
Name: id, dtype: float64

Date range:
Earliest review: 2009-05-25
Latest review: 2025-08-02

Missing comments:
260

Sample reviews:
['Fantastic to get to live in Williamsburg at A and A’s place! We loved the surroundings and our hosts. They were sweet, nice, helpful, inviting people! Even the neighbours were nice. We had good talks on the wonderful patio.<br/>The room and private bathrooms was fine and clean and with a good WiFi ;-) The bed was a little soft... but equipped with tons of good pillows :-) <br/>It’s SO easy (short walk) to get to the subway - and only a short train ride to Manhattan. We enjoyed Graham Avenue’s shops, restaurants, a great barber shop - and the quiet neighbourhood. ', 'We loved staying here!  It was beyond what we expected, clean, comfortable and so well done!  Lo

## Clean and merge with listings

In [51]:
# drop missing comments
reviews = reviews.dropna(subset=['comments'])

# clean HTML tags
reviews['comments'] = reviews['comments'].str.replace('<br/>', ' ', regex=False)
reviews['comments'] = reviews['comments'].str.replace('<[^>]+>', ' ', regex=True)

# merge with listings
merged = reviews.merge(
    listings_clean[['id', 'instant_bookable', 'neighbourhood_group_cleansed',
                    'room_type', 'estimated_occupancy_l365d', 'price']],
    left_on='listing_id',
    right_on='id',
    how='inner'
)
print(f'Reviews after merge: {merged.shape}')
print(f'Unique listings in merged: {merged["listing_id"].nunique()}')

# review stats
merged['review_stats'] = merged['comments'].str.len()
print('Review stats:')
print(merged['review_stats'].describe())

# reviews by year
merged['date'] = pd.to_datetime(merged['date'])
merged['year'] = merged['date'].dt.year
print('\nReviews by year:')
print(merged['year'].value_counts().sort_index())

Reviews after merge: (936325, 12)
Unique listings in merged: 23976
Review stats:
count    936325.000000
mean        253.214663
std         251.741871
min           1.000000
25%          86.000000
50%         183.000000
75%         336.000000
max        5921.000000
Name: review_stats, dtype: float64

Reviews by year:
year
2009        43
2010       416
2011      1632
2012      3229
2013      5865
2014     11264
2015     23056
2016     39896
2017     54965
2018     78602
2019    101845
2020     39131
2021     80502
2022    137689
2023    157147
2024    124488
2025     76555
Name: count, dtype: int64


### Verify the merge

In [52]:
print('\nColumns Before Merge:')
print(reviews.columns.tolist())
print('\nMerged Columns:')
print(merged.columns.tolist())
print(merged[['listing_id', 'id_x', 'id_y']].head())

# null check
print('\nNull values in merged dataset:')
print(merged.isnull().sum())

# reviews per listing -- are reviews concentrated on a few listings?
reviews_per_listing = merged.groupby('listing_id')['id_x'].count()
print(reviews_per_listing.describe())
print(f'\nTop 5 most reviewed listings:')
print(reviews_per_listing.sort_values(ascending=False).head())


Columns Before Merge:
['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']

Merged Columns:
['listing_id', 'id_x', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'id_y', 'instant_bookable', 'neighbourhood_group_cleansed', 'room_type', 'estimated_occupancy_l365d', 'price', 'review_stats', 'year']
   listing_id       id_x  id_y
0        2539   55688172  2539
1        2539   97474898  2539
2        2539  105340344  2539
3        2539  133131670  2539
4        2539  138349776  2539

Null values in merged dataset:
listing_id                      0
id_x                            0
date                            0
reviewer_id                     0
reviewer_name                   4
comments                        0
id_y                            0
instant_bookable                0
neighbourhood_group_cleansed    0
room_type                       0
estimated_occupancy_l365d       0
price                           0
review_stats                    0
year             

Review distribution is highly skewed -- median 11 reviews per listing 
but max 3,517. Listing-level sentiment will be aggregated as a mean 
across all reviews, so heavily reviewed listings will have more stable 
sentiment estimates than listings with only 1-2 reviews.

## Export for sentiment scoring

Sentiment scoring runs on Colab (free GPU) rather than locally -- 
saving the merged dataset here to upload there.

In [53]:
# save reviews for Colab sentiment scoring
merged.to_csv('data/reviews_merged.csv', index=False)
print(f'Saved {len(merged)} reviews to reviews_merged.csv')

Saved 936325 reviews to reviews_merged.csv


## Sentiment scoring

Ran HuggingFace's DistilBERT sentiment model on Colab. Sampled 800,000 
of the 936,325 total reviews due to free-tier GPU session time limits 
-- still covers 98% of listings. Loading the results back in here.

In [54]:
sentiment = pd.read_csv('data/sentiment_scores.csv')
print(f'Loaded {len(sentiment)} sentiment scores')
print(sentiment.head())
print(sentiment['sentiment_label'].value_counts())

Loaded 800000 sentiment scores
                  id_x  listing_id  \
0   660801654348790896    16396241   
1  1310235268241746719     4268286   
2            707590208    27598707   
3   677517481374761544    49274169   
4   628222255433284746    54247180   

                                            comments sentiment_label  \
0  Loved our stay here. Very close to great spots...        POSITIVE   
1  Stayed there last week for 4nights. The locati...        POSITIVE   
2  Great stay at the Freehand. The room was very ...        POSITIVE   
3  We had a very nice stay here.  Host responses ...        POSITIVE   
4  The hosts were accommodating. Naomi was out of...        POSITIVE   

   sentiment_score  
0         0.999681  
1         0.998254  
2         0.999803  
3         0.999661  
4         0.999417  
sentiment_label
POSITIVE    700250
NEGATIVE     99750
Name: count, dtype: int64


### Merging sentiment back to review text

The first version of this merge used `listing_id` and produced 81 
million rows instead of the expected ~800,000 -- a cross-join. Since 
listing_id repeats across every review for a listing, matching on it 
alone paired every review with every sentiment score for that same 
listing, rather than each review with its own specific score.

Fixed by keeping a unique review ID (`id_x`) alongside the sentiment 
label when saving results from Colab, and merging on that instead -- 
a clean 1-to-1 match.

In [55]:
merged_sentiment = merged.merge(
    sentiment[['id_x', 'sentiment_label', 'sentiment_score']],
    on='id_x',
    how='inner'
)
print(f'Merged rows: {len(merged_sentiment)}')
print(f'Sentiment rows: {len(sentiment)}')

Merged rows: 800000
Sentiment rows: 800000


## Sentiment vs occupancy

Aggregating sentiment to the listing level and checking whether it 
relates to occupancy, Instant Book status, room type, and borough.

In [56]:
# aggregate sentiment to listing level
listing_sentiment = sentiment.groupby('listing_id').agg(
    avg_sentiment_score=('sentiment_score', 'mean'),
    pct_positive=('sentiment_label', lambda x: (x == 'POSITIVE').mean()),
    review_count=('sentiment_label', 'count')
).reset_index()
print(f'Listings with sentiment: {len(listing_sentiment)}')
print(listing_sentiment.describe())

# merge with listings
listings_sentiment = listings_clean.merge(
    listing_sentiment,
    left_on='id',
    right_on='listing_id',
    how='left'
)
print(f'Listings with sentiment: {listings_sentiment["avg_sentiment_score"].notna().sum()}')
print(f'Listings without sentiment: {listings_sentiment["avg_sentiment_score"].isna().sum()}')

# correlation with occupancy
corr = listings_sentiment[['avg_sentiment_score', 'pct_positive', 
                            'estimated_occupancy_l365d']].corr()
print('\nCorrelation with occupancy:')
print(corr['estimated_occupancy_l365d'])

Listings with sentiment: 23453
         listing_id  avg_sentiment_score  pct_positive  review_count
count  2.345300e+04         23453.000000  23453.000000  23453.000000
mean   3.204678e+17             0.984889      0.877891     34.110775
std    4.537091e+17             0.030427      0.187513     69.074120
min    2.539000e+03             0.512411      0.000000      1.000000
25%    1.772700e+07             0.981106      0.833333      3.000000
50%    4.251647e+07             0.994275      0.935484     10.000000
75%    7.477222e+17             0.999581      1.000000     37.000000
max    1.409437e+18             0.999894      1.000000   3044.000000
Listings with sentiment: 23453
Listings without sentiment: 13160

Correlation with occupancy:
avg_sentiment_score         -0.038939
pct_positive                -0.044610
estimated_occupancy_l365d    1.000000
Name: estimated_occupancy_l365d, dtype: float64


Sentiment shows weak negative correlation with occupancy (-0.038 to -0.045).
High occupancy listings do not necessarily have better sentiment -- 
possibly because popular listings attract more reviews including 
more critical ones, or because structured features like review scores 
already capture listing quality more precisely than text sentiment.

### Sentiment by Instant Book, room type, and borough

In [57]:
# Sentiment by Instant Book status
ib_sentiment = listings_sentiment.groupby('instant_bookable').agg(
    avg_sentiment=('avg_sentiment_score', 'mean'),
    pct_positive=('pct_positive', 'mean'),
    count=('id', 'count')
).reset_index()

print(ib_sentiment)

   instant_bookable  avg_sentiment  pct_positive  count
0             False       0.985644      0.886891  29108
1              True       0.981512      0.837611   7505


In [58]:
# Sentiment by room type
room_sentiment = listings_sentiment.groupby('room_type').agg(
    avg_sentiment=('avg_sentiment_score', 'mean'),
    pct_positive=('pct_positive', 'mean'),
    avg_occupancy=('estimated_occupancy_l365d', 'mean'),
    count=('id', 'count')
).reset_index()

print(room_sentiment)

         room_type  avg_sentiment  pct_positive  avg_occupancy  count
0  Entire home/apt       0.985920      0.886571      49.368466  19706
1       Hotel room       0.984618      0.839058      10.496945    491
2     Private room       0.983625      0.867240      44.635401  16237
3      Shared room       0.979010      0.868236      34.072626    179


Room type sentiment patterns matches with Project 1 occupancy findings --
entire homes show highest sentiment and highest occupancy, while 
hotel rooms show lowest on both metrics. At borough level, Brooklyn and 
Queens show strong sentiment consistent with their significant occupancy 
lifts found in Project 1 subgroup analysis. Manhattan's lower sentiment 
despite high demand suggests guests have higher expectations in premium 
markets.

In [59]:
# Sentiment by borough
borough_sentiment = listings_sentiment.groupby('neighbourhood_group_cleansed').agg(
    avg_sentiment=('avg_sentiment_score', 'mean'),
    pct_positive=('pct_positive', 'mean'),
    avg_occupancy=('estimated_occupancy_l365d', 'mean'),
    count=('id', 'count')
).reset_index()

print(borough_sentiment)

  neighbourhood_group_cleansed  avg_sentiment  pct_positive  avg_occupancy  \
0                        Bronx       0.984605      0.880700      51.835720   
1                     Brooklyn       0.987013      0.901141      48.081224   
2                    Manhattan       0.983088      0.855264      43.055454   
3                       Queens       0.984292      0.878422      51.914329   
4                Staten Island       0.987336      0.897582      64.591160   

   count  
0   1187  
1  13432  
2  16356  
3   5276  
4    362  


## Topic analysis

What do guests actually praise or complain about? Splitting reviews 
by sentiment label and looking at word frequency.

In [60]:
positive_reviews = merged_sentiment[
    merged_sentiment['sentiment_label'] == 'POSITIVE']['comments']
negative_reviews = merged_sentiment[
    merged_sentiment['sentiment_label'] == 'NEGATIVE']['comments']
print(f'Positive reviews: {len(positive_reviews)}')
print(f'Negative reviews: {len(negative_reviews)}')

print('\nTop words in POSITIVE reviews:')
print(get_top_words(positive_reviews))
print('\nTop words in NEGATIVE reviews:')
print(get_top_words(negative_reviews))

Positive reviews: 700250
Negative reviews: 99750

Top words in POSITIVE reviews:
[('great', 406742), ('place', 362498), ('stay', 356293), ('clean', 211505), ('location', 202767), ('apartment', 192902), ('host', 168971), ('nice', 145740), ('again', 128515), ('recommend', 121764), ('comfortable', 121603), ('everything', 114087), ('room', 112547), ('subway', 107372), ('all', 104025), ('easy', 102983), ('definitely', 102862), ('good', 99880), ('close', 97666), ('really', 95863)]

Top words in NEGATIVE reviews:
[('muy', 31132), ('que', 31010), ('und', 26647), ('est', 26141), ('très', 23840), ('die', 19916), ('nous', 19447), ('bien', 17103), ('room', 17098), ('place', 16840), ('para', 16464), ('con', 16174), ('sehr', 14871), ('stay', 14369), ('ist', 14286), ('pour', 13539), ('apartment', 12985), ('una', 12942), ('manhattan', 12429), ('appartement', 12386)]


### Language limitation 

Negative review word frequency is dominated by non-English words 
(muy, que, und, très, die, sehr) rather than genuine complaints. 
DistilBERT was trained on English text only -- when it encounters 
Spanish, French, or German reviews it appears to default toward 
NEGATIVE classification, likely because it can't recognize positive 
sentiment expressions in other languages.

This means the negative sentiment label is conflating two different 
things: genuinely negative English reviews, and non-English reviews 
the model can't properly classify. Topic analysis on negative reviews 
needs to filter to English text first to get a meaningful signal.

In [61]:
def is_likely_english(text):
    english_markers = {'the', 'was', 'and', 'very', 'stay', 'great',
                       'place', 'clean', 'nice', 'room'}
    words = set(str(text).lower().split())
    return len(words & english_markers) >= 2

merged_sentiment['is_english'] = merged_sentiment['comments'].apply(is_likely_english)
english_reviews = merged_sentiment[merged_sentiment['is_english']]

positive_reviews_en = english_reviews[
    english_reviews['sentiment_label'] == 'POSITIVE']['comments']
negative_reviews_en = english_reviews[
    english_reviews['sentiment_label'] == 'NEGATIVE']['comments']

print(f'English positive reviews: {len(positive_reviews_en)}')
print(f'English negative reviews: {len(negative_reviews_en)}')

print('\nTop words in POSITIVE (English only):')
for word, count in get_top_words(positive_reviews_en):
    print(f'{word}: {count}')

print('\nTop words in NEGATIVE (English only):')
for word, count in get_top_words(negative_reviews_en):
    print(f'{word}: {count}')

English positive reviews: 587332
English negative reviews: 29226

Top words in POSITIVE (English only):
great: 379479
place: 348848
stay: 341499
clean: 206455
apartment: 187600
location: 185954
host: 158236
nice: 141168
again: 123336
comfortable: 117975
recommend: 115241
room: 110939
everything: 108724
subway: 105185
all: 101544
easy: 99383
definitely: 98645
close: 94639
really: 92870
perfect: 89083

Top words in NEGATIVE (English only):
room: 16844
place: 16078
stay: 14039
apartment: 11330
host: 10051
great: 9798
location: 9628
out: 9206
all: 9082
clean: 8860
when: 8851
only: 8277
which: 7931
one: 7914
night: 7491
good: 7375
bathroom: 7139
also: 7061
just: 6937
bed: 6859


### From single words to phrases

Single words weren't specific enough to tell what was actually being 
complained about -- "room" or "bathroom" alone don't say if something 
was dirty, small, or missing. Trigrams surface fuller phrases instead.

In [62]:
print('Top trigrams in NEGATIVE (English only):')
for phrase, count in get_top_trigrams(negative_reviews_en):
    print(f'{phrase}: {count}')

Top trigrams in NEGATIVE (English only):
the room was: 1684
the place was: 1548
the apartment is: 1478
we had to: 1454
in the room: 1450
the apartment was: 1223
there was no: 1204
the place is: 1167
place to stay: 1131
if you are: 1112
the location is: 1087
in the bathroom: 1078
if you re: 1077
close to the: 1061
in the apartment: 1038
to the subway: 957
and it was: 953
there is no: 944
the host was: 943
the living room: 927
one of the: 923
the room is: 918
for the price: 781
but it was: 766
the bed was: 762


### What guests say is missing

Pulling out exactly what follows "there was no" / "there is no" in 
negative reviews, to get concrete, fixable complaints instead of just 
frequent words.

In [63]:
no_phrases = negative_reviews_en[negative_reviews_en.str.contains(
    'there was no|there is no', case=False, na=False)]
missing_items = []
for text in no_phrases:
    matches = re.findall(r'there (?:was|is) no (\w+)', str(text).lower())
    missing_items.extend(matches)

print('Most commonly missing items (from "there was/is no ___"):')
print(Counter(missing_items).most_common(15))

Most commonly missing items (from "there was/is no ___"):
[('way', 83), ('elevator', 70), ('ac', 53), ('hot', 53), ('parking', 42), ('air', 40), ('tv', 38), ('place', 36), ('microwave', 35), ('one', 32), ('heat', 32), ('lock', 31), ('hand', 31), ('privacy', 30), ('soap', 27)]


### Topic analysis findings

Positive reviews consistently praise cleanliness, location, host 
responsiveness, and comfort. Leading guests to frequently signal intent to return.

Negative reviews center on specific missing amenities rather than 
harsh complaints -- extracting phrases following "there was/is no" 
reveals the most common gaps guests report:

1. No elevator (70 mentions) -- building accessibility
2. No AC / no hot water (106 combined mentions) 
3. No parking (42 mentions)
4. No TV / no microwave/ no soap -- basic amenity gaps
5. No privacy, no lock -- security and comfort concerns

This gives hosts a concrete action list: prioritizing AC/heating 
reliability and clear amenity disclosure (elevator access, parking 
availability) could meaningfully reduce negative reviews.

## Novelty check

Does sentiment tell us anything Project 1's variables didn't already 
capture? Comparing an OLS model with just Project 1's controls against 
the same model with sentiment added.

In [64]:
# prepare data -- listings with sentiment and all Project 1 control variables
novelty_df = listings_sentiment.dropna(subset=[
    'avg_sentiment_score', 'price', 'room_type', 
    'neighbourhood_group_cleansed', 'review_scores_rating',
    'instant_bookable', 'estimated_occupancy_l365d'
]).copy()
print(f'Listings used in novelty check: {len(novelty_df)}')

# model 1 -- Project 1 controls only (no sentiment)
model_base = smf.ols(
    'estimated_occupancy_l365d ~ price + C(room_type) + '
    'C(neighbourhood_group_cleansed) + review_scores_rating + '
    'C(instant_bookable)',
    data=novelty_df
).fit()
print('=== Model 1: Project 1 controls only ===')
print(f'R-squared: {model_base.rsquared:.4f}')

# model 2 -- Project 1 controls + sentiment
model_full = smf.ols(
    'estimated_occupancy_l365d ~ price + C(room_type) + '
    'C(neighbourhood_group_cleansed) + review_scores_rating + '
    'C(instant_bookable) + avg_sentiment_score',
    data=novelty_df
).fit()
print('\n=== Model 2: Project 1 controls + sentiment ===')
print(f'R-squared: {model_full.rsquared:.4f}')
print(f'\nSentiment coefficient: {model_full.params["avg_sentiment_score"]:.4f}')
print(f'Sentiment p-value: {model_full.pvalues["avg_sentiment_score"]:.4f}')
print(f'R-squared improvement: {model_full.rsquared - model_base.rsquared:.4f}')

Listings used in novelty check: 23453
=== Model 1: Project 1 controls only ===
R-squared: 0.0366

=== Model 2: Project 1 controls + sentiment ===
R-squared: 0.0384

Sentiment coefficient: -131.0873
Sentiment p-value: 0.0000
R-squared improvement: 0.0017


### Does sentiment tell us anything new?

After controlling for price, room type, borough, review score, and 
Instant Book status, sentiment is still statistically significant 
(p < 0.0001) -- so it's not just noise. But it barely moves the needle 
practically. Adding sentiment only improves R-squared from 0.037 to 
0.038, a 0.17 percentage point gain.

The sentiment coefficient is also negative, which is a bit 
counterintuitive -- higher sentiment is weakly linked to lower 
occupancy once everything else is accounted for. This lines up with 
an earlier finding: the most booked listings tend to rack up more 
reviews overall, including more mixed ones, which pulls their average 
sentiment down slightly.

**Takeaway**: Sentiment adds a real but small amount of predictive 
signal on top of what Project 1 already found. Review scores and the 
other structured features are doing most of the heavy lifting in 
explaining occupancy. Where sentiment is genuinely useful is 
qualitative -- it tells you *why* guests feel the way they do, not 
just *how much* they liked a listing.

## Business recommendation

Guest sentiment is overwhelmingly positive (87.5%), and tracks 
reasonably well with the structured features from Project 1 -- entire 
homes and Brooklyn listings show both higher sentiment and higher 
occupancy, while hotel rooms and Manhattan show lower on both. But 
sentiment only adds a small amount of new predictive information on 
top of price, room type, borough, and review score -- its real value 
here isn't prediction, it's explanation. It tells us why guests feel 
the way they do, which the numbers alone can't.

**What guests actually complain about:**

Pulling out what's missing from negative reviews, the most common gaps 
are concrete and fixable, not vague dissatisfaction:
- No air conditioning or hot water
- No elevator
- No parking
- Missing basic amenities like a TV or microwave or soap
- Privacy and lock concerns

**How this could be used:**
- Host guidance -- prioritize AC/heating reliability and clearly 
  disclose building access (elevator, parking) in the listing, since 
  these are the most common gaps between guest expectation and reality
- Review moderation -- sentiment alone isn't worth over-indexing on for 
  ranking, but it's a useful early warning system for identifying 
  specific, fixable listing issues
- Search and recommendations -- the recommender in Project 2 doesn't 
  need sentiment as an input, since structured features already 
  capture what matters most for matching guests to listings

**Connecting all three projects:**

Project 1 found that Instant Book -- which removes the host approval 
step from booking -- is associated with higher occupancy, though the 
effect may partly reflect underlying listing quality rather than the 
booking mechanism alone. Project 2 built a recommender using the features Project 1 identified 
as most important, and showed that blending guest preferences with 
real occupancy data meaningfully improves recommendation quality. 
Project 3 confirms those structured features already capture most of 
what drives occupancy -- guest sentiment adds real but modest 
additional signal, and its main value is surfacing concrete, 
actionable complaints rather than improving prediction.